# SITS - Clustering de Padroes Temporais

Este notebook aplica Deep Temporal Clustering (DTC) para identificar subpadroes dentro de cada classe.

**Metodologia:**
1. Extrair pixels da imagem classificada com amostragem espacial em grid
2. Calcular indice (NDVI ou outro) de cada pixel
3. Treinar DTC para diferentes valores de K
4. Analisar thresholds de probabilidade para filtrar ruido
5. Gerar mapas espaciais dos clusters

**Saidas:**
- Padroes temporais (curvas medias)
- Mapas de clusters (GeoTIFF)
- Mapas de confianca

---
## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
import json

# Imports do modulo clustering
from sits.clustering import (
    # Extracao de dados
    extract_pixels_spatial_grid,
    save_samples,
    load_samples,
    # Modelos
    DTCAutoencoder,
    DTCAutoencoderWithAttention,
    # Treinamento
    pretrain_autoencoder,
    finetune_dtc,
    # Analise
    analyze_thresholds,
    create_threshold_summary_df,
    compute_cluster_profiles,
    find_best_configuration,
    save_k_results,
    save_comparison_results,
)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# =============================================================================
# CONFIGURACAO - EDITE AQUI
# =============================================================================

# Caminhos
SESSION_PATH = Path("../sessions/seu_projeto")
EXPERIMENT_NAME = "exp_v1"

# Imagens
IMAGE_PATH = Path("../data/stack_temporal.tif")  # Imagem original
CLASSIFICATION_PATH = SESSION_PATH / "training" / EXPERIMENT_NAME / "inference" / "classificacao.tif"

# Classe a analisar (indice)
CLASSE_ANALISE = 0

# Mapeamento de classes
CLASSE_NOMES = {
    0: "classe_0",
    1: "classe_1",
    2: "classe_2",
    # Adicione suas classes
}

# Parametros de extracao
MAX_PIXELS = 1_000_000  # Maximo de amostras
GRID_SIZE = 50          # Tamanho do grid para amostragem espacial

# Parametros de clustering
K_RANGE = [2, 3, 4, 5]  # Valores de K para testar
BATCH_SIZE = 4096
PRETRAIN_EPOCHS = 100
FINETUNE_EPOCHS = 200
THRESHOLDS = [0.6, 0.7, 0.8, 0.9]  # Thresholds de probabilidade

# Nomes dos timesteps (para visualizacao)
TIMESTEP_NAMES = ['T01', 'T02', 'T03', 'T04', 'T05', 'T06', 
                  'T07', 'T08', 'T09', 'T10', 'T11', 'T12']

# Diretorio de saida
CLASSE_NOME = CLASSE_NOMES.get(CLASSE_ANALISE, f"classe_{CLASSE_ANALISE}")
OUTPUT_DIR = SESSION_PATH / "clustering" / CLASSE_NOME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("="*60)
print("CONFIGURACAO")
print("="*60)
print(f"Classe: {CLASSE_NOME} (idx={CLASSE_ANALISE})")
print(f"Max pixels: {MAX_PIXELS:,}")
print(f"K range: {K_RANGE}")
print(f"Thresholds: {THRESHOLDS}")
print(f"Output: {OUTPUT_DIR}")
print("="*60)

---
## 2. Extracao de Pixels

In [ ]:
# Verificar se samples ja existem
samples_file = OUTPUT_DIR / "samples.npz"

if samples_file.exists():
    print("Amostras ja extraidas. Carregando...")
    ndvi, rows, cols, metadata = load_samples(OUTPUT_DIR)
else:
    print(f"Extraindo pixels com amostragem espacial (grid {GRID_SIZE}x{GRID_SIZE})...")
    ndvi, rows, cols, metadata = extract_pixels_spatial_grid(
        image_path=str(IMAGE_PATH),
        classification_path=str(CLASSIFICATION_PATH),
        target_class=CLASSE_ANALISE,
        max_pixels=MAX_PIXELS,
        grid_size=GRID_SIZE
    )
    
    # Salvar
    save_samples(OUTPUT_DIR, ndvi, rows, cols, metadata)

print(f"\nAmostras: {len(ndvi):,}")
print(f"NDVI range: [{ndvi.min():.3f}, {ndvi.max():.3f}]")

In [ ]:
# Visualizacao exploratoria
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Curvas NDVI (amostra)
ax1 = axes[0]
n_plot = min(500, len(ndvi))
idx_plot = np.random.choice(len(ndvi), n_plot, replace=False)
for i in idx_plot:
    ax1.plot(TIMESTEP_NAMES, ndvi[i], alpha=0.1, color='green')
ax1.plot(TIMESTEP_NAMES, ndvi.mean(axis=0), 'k-', linewidth=2, label='Media')
ax1.set_xlabel('Timestep')
ax1.set_ylabel('NDVI')
ax1.set_title(f'Curvas NDVI - {CLASSE_NOME} (n={len(ndvi):,})')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.tick_params(axis='x', rotation=45)

# Plot 2: Histograma NDVI medio
ax2 = axes[1]
mean_ndvi = ndvi.mean(axis=1)
ax2.hist(mean_ndvi, bins=50, edgecolor='black', alpha=0.7)
ax2.axvline(mean_ndvi.mean(), color='red', linestyle='--', label=f'Media: {mean_ndvi.mean():.3f}')
ax2.set_xlabel('NDVI Medio')
ax2.set_ylabel('Frequencia')
ax2.set_title('Distribuicao do NDVI Medio')
ax2.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'exploratory_ndvi.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Pre-treino do Autoencoder

In [ ]:
# O pre-treino eh K-agnostico - aprende a compressao dos dados
# Depois reutilizamos para cada valor de K

print("="*60)
print("PRE-TREINO DO AUTOENCODER")
print("="*60)

pretrain_file = OUTPUT_DIR / "pretrained_model.pt"

if pretrain_file.exists():
    print("Modelo pre-treinado encontrado. Carregando...")
    checkpoint = torch.load(pretrain_file, map_location=device)
    use_attention = checkpoint['config'].get('use_attention', False)
    
    if use_attention:
        pretrained_model = DTCAutoencoderWithAttention(
            input_dim=1, seq_len=len(TIMESTEP_NAMES),
            hidden_dim=checkpoint['config']['hidden_dim'],
            latent_dim=checkpoint['config']['latent_dim']
        ).to(device)
    else:
        pretrained_model = DTCAutoencoder(
            input_dim=1, seq_len=len(TIMESTEP_NAMES),
            hidden_dim=checkpoint['config']['hidden_dim'],
            latent_dim=checkpoint['config']['latent_dim']
        ).to(device)
    
    pretrained_model.load_state_dict(checkpoint['model'])
    pretrain_embeddings = np.load(OUTPUT_DIR / "pretrain_embeddings.npy")
    print(f"Embeddings: {pretrain_embeddings.shape}")
else:
    print("Treinando novo modelo...")
    pretrained_model, pretrain_embeddings = pretrain_autoencoder(
        ndvi=ndvi,
        device=device,
        pretrain_epochs=PRETRAIN_EPOCHS,
        batch_size=BATCH_SIZE,
        hidden_dim=32,
        latent_dim=8,
        use_attention=True,
        verbose=True
    )
    
    # Salvar modelo pre-treinado
    torch.save({
        'model': pretrained_model.state_dict(),
        'config': {
            'hidden_dim': 32, 
            'latent_dim': 8,
            'use_attention': True
        }
    }, pretrain_file)
    np.save(OUTPUT_DIR / "pretrain_embeddings.npy", pretrain_embeddings)
    print(f"\nModelo salvo em: {pretrain_file}")

In [ ]:
# Visualizacao dos pesos de atencao (se disponivel)
if hasattr(pretrained_model, 'get_attention_weights'):
    print("Calculando pesos de atencao...")
    
    n_sample = min(10000, len(ndvi))
    idx_sample = np.random.choice(len(ndvi), n_sample, replace=False)
    X_sample = torch.FloatTensor(ndvi[idx_sample, :, np.newaxis]).to(device)
    
    pretrained_model.eval()
    attn_weights = pretrained_model.get_attention_weights(X_sample).cpu().numpy()
    
    mean_attention = attn_weights.mean(axis=0)
    std_attention = attn_weights.std(axis=0)
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 5))
    
    bars = ax.bar(TIMESTEP_NAMES, mean_attention, yerr=std_attention, 
                  capsize=3, color='steelblue', alpha=0.8, edgecolor='black')
    
    # Destacar timesteps mais importantes
    top_idx = np.argsort(mean_attention)[-3:]
    for i in top_idx:
        bars[i].set_color('darkgreen')
    
    ax.set_xlabel('Timestep')
    ax.set_ylabel('Peso de Atencao')
    ax.set_title('Importancia de Cada Timestep para o Clustering')
    ax.grid(True, alpha=0.3, axis='y')
    ax.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'attention_weights.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nTimesteps mais importantes:")
    for i in np.argsort(mean_attention)[::-1][:5]:
        print(f"  {TIMESTEP_NAMES[i]}: {mean_attention[i]:.4f}")
else:
    print("Modelo sem atencao - visualizacao nao disponivel.")

---
## 4. Fine-tuning para Cada K

In [ ]:
k_results = {}

for k in K_RANGE:
    print(f"\n{'='*60}")
    print(f"FINE-TUNING K={k}")
    print("="*60)
    
    k_dir = OUTPUT_DIR / f"k{k}"
    
    # Verificar se ja existe
    if (k_dir / "results.npz").exists():
        print(f"K={k} ja treinado. Carregando...")
        data = np.load(k_dir / "results.npz")
        labels = data['labels']
        probs = data['probs']
        embeddings = data['embeddings']
        model_state = None
    else:
        # Fine-tuning
        labels, probs, embeddings, model_state = finetune_dtc(
            ndvi=ndvi,
            pretrained_model=pretrained_model,
            embeddings=pretrain_embeddings,
            n_clusters=k,
            device=device,
            finetune_epochs=FINETUNE_EPOCHS,
            batch_size=BATCH_SIZE,
            verbose=True
        )
    
    # Analisar thresholds
    print(f"\nAnalisando thresholds...")
    threshold_results = analyze_thresholds(
        probs=probs,
        labels=labels,
        embeddings=embeddings,
        ndvi=ndvi,
        thresholds=THRESHOLDS
    )
    
    # Perfis dos clusters
    cluster_profiles = compute_cluster_profiles(ndvi, labels, probs, threshold=0.7)
    
    # Salvar
    if model_state is not None:
        save_k_results(
            output_dir=str(OUTPUT_DIR),
            k=k,
            model_state=model_state,
            labels=labels,
            probs=probs,
            embeddings=embeddings,
            threshold_results=threshold_results,
            cluster_profiles=cluster_profiles
        )
    
    # Armazenar
    k_results[k] = {
        'labels': labels,
        'probs': probs,
        'embeddings': embeddings,
        'thresholds': threshold_results,
        'profiles': cluster_profiles
    }
    
    # Resumo
    df_summary = create_threshold_summary_df(threshold_results)
    print(f"\nResumo K={k}:")
    print(df_summary.to_string(index=False))

print("\n" + "="*60)
print("TREINAMENTO CONCLUIDO!")
print("="*60)

---
## 5. Comparacao entre K

In [ ]:
import pandas as pd

# Criar tabela comparativa
comparison_rows = []
for k in K_RANGE:
    for threshold, t_data in k_results[k]['thresholds'].items():
        if t_data['metrics'] is not None:
            comparison_rows.append({
                'K': k,
                'Threshold': threshold,
                'Amostras': t_data['n_filtered'],
                '% Removido': t_data['pct_removed'],
                'Silhouette': round(t_data['metrics']['silhouette'], 4),
                'Calinski': round(t_data['metrics']['calinski_harabasz'], 0),
                'Davies': round(t_data['metrics']['davies_bouldin'], 4)
            })

df_comparison = pd.DataFrame(comparison_rows)
print("Comparacao completa:")
print(df_comparison.to_string(index=False))

In [ ]:
# Grafico: metricas vs K
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, metric, title in zip(axes, 
                              ['Silhouette', 'Calinski', 'Davies'],
                              ['Silhouette (maior = melhor)', 
                               'Calinski-Harabasz (maior = melhor)',
                               'Davies-Bouldin (menor = melhor)']):
    for threshold in THRESHOLDS:
        df_t = df_comparison[df_comparison['Threshold'] == threshold]
        ax.plot(df_t['K'], df_t[metric], 'o-', label=f't={threshold}')
    
    ax.set_xlabel('K (numero de clusters)')
    ax.set_ylabel(metric)
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'k_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Encontrar melhor configuracao
best_config = find_best_configuration(k_results, metric='silhouette', threshold=0.7)

if best_config:
    print("\n" + "="*60)
    print("MELHOR CONFIGURACAO")
    print("="*60)
    print(f"K = {best_config['best_k']}")
    print(f"Threshold = {best_config['best_threshold']}")
    print(f"Silhouette = {best_config['best_score']:.4f}")
    
    # Salvar
    save_comparison_results(str(OUTPUT_DIR), k_results, best_config)
else:
    print("Nao foi possivel determinar melhor configuracao.")

---
## 6. Analise do Melhor K

In [ ]:
# Selecionar melhor K
BEST_K = best_config['best_k'] if best_config else K_RANGE[0]
BEST_THRESHOLD = 0.7

print(f"Analisando K={BEST_K} com threshold={BEST_THRESHOLD}")

labels = k_results[BEST_K]['labels']
probs = k_results[BEST_K]['probs']
profiles = k_results[BEST_K]['profiles']

# Filtrar por threshold
max_probs = probs.max(axis=1)
mask_conf = max_probs >= BEST_THRESHOLD

print(f"Amostras totais: {len(labels):,}")
print(f"Amostras confiaveis (prob >= {BEST_THRESHOLD}): {mask_conf.sum():,} ({mask_conf.mean()*100:.1f}%)")

In [ ]:
# Visualizar perfis dos clusters
n_clusters = BEST_K
fig, axes = plt.subplots(1, n_clusters, figsize=(5*n_clusters, 5))
if n_clusters == 1:
    axes = [axes]

colors = plt.cm.tab10(np.linspace(0, 1, n_clusters))

for c, ax in enumerate(axes):
    if c not in profiles:
        continue
        
    profile = profiles[c]
    mean_curve = np.array(profile['mean'])
    std_curve = np.array(profile['std'])
    
    ax.fill_between(TIMESTEP_NAMES, mean_curve - std_curve, mean_curve + std_curve, 
                    alpha=0.3, color=colors[c])
    ax.plot(TIMESTEP_NAMES, mean_curve, 'o-', color=colors[c], linewidth=2)
    
    ax.set_ylim(-0.2, 1.0)
    ax.set_xlabel('Timestep')
    ax.set_ylabel('NDVI')
    ax.set_title(f"Cluster {c}\n"
                 f"n={profile['n_samples']:,}\n"
                 f"Pico: {profile.get('peak_month_name', 'N/A')}")
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

plt.suptitle(f'Perfis dos Clusters - {CLASSE_NOME} (K={BEST_K})', fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'k{BEST_K}' / 'cluster_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Distribuicao de probabilidades
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Histograma de probabilidade maxima
ax1 = axes[0]
ax1.hist(max_probs, bins=50, edgecolor='black', alpha=0.7)
for t in THRESHOLDS:
    ax1.axvline(t, color='red', linestyle='--', alpha=0.5)
ax1.axvline(BEST_THRESHOLD, color='red', linestyle='-', linewidth=2, label=f'Threshold={BEST_THRESHOLD}')
ax1.set_xlabel('Probabilidade Maxima')
ax1.set_ylabel('Frequencia')
ax1.set_title('Distribuicao de Confianca')
ax1.legend()

# Amostras por cluster
ax2 = axes[1]
labels_conf = labels[mask_conf]
unique, counts = np.unique(labels_conf, return_counts=True)
ax2.bar(unique, counts, color=[colors[c] for c in unique], edgecolor='black')
ax2.set_xlabel('Cluster')
ax2.set_ylabel('Numero de Amostras')
ax2.set_title(f'Amostras por Cluster (threshold={BEST_THRESHOLD})')
for i, (u, c) in enumerate(zip(unique, counts)):
    ax2.text(u, c + max(counts)*0.01, f'{c:,}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'k{BEST_K}' / 'probability_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Geracao de Mapas

In [ ]:
import rasterio

# Carregar metadados da imagem original
with rasterio.open(IMAGE_PATH) as src:
    height, width = src.height, src.width
    crs = src.crs
    transform = src.transform

print(f"Dimensoes da imagem: {width} x {height}")
print(f"CRS: {crs}")

In [ ]:
# Criar mapa de clusters
cluster_map = np.full((height, width), -1, dtype=np.int8)
cluster_map[rows, cols] = labels

# Marcar incertos
uncertain_mask = ~mask_conf
cluster_map[rows[uncertain_mask], cols[uncertain_mask]] = -2

# Mapa de confianca
confidence_map = np.full((height, width), 0, dtype=np.float32)
confidence_map[rows, cols] = max_probs

print(f"Valores do mapa: {np.unique(cluster_map)}")
print(f"  -1 = fora da classe")
print(f"  -2 = incerto (prob < {BEST_THRESHOLD})")
print(f"  0-{BEST_K-1} = clusters")

In [ ]:
# Salvar mapas como GeoTIFF
output_dir_maps = OUTPUT_DIR / 'output'
output_dir_maps.mkdir(exist_ok=True)

# Mapa de clusters
cluster_tif_path = output_dir_maps / f'cluster_map_k{BEST_K}.tif'
with rasterio.open(
    cluster_tif_path, 'w',
    driver='GTiff',
    height=height,
    width=width,
    count=1,
    dtype=np.int8,
    crs=crs,
    transform=transform,
    nodata=-1,
    compress='lzw'
) as dst:
    dst.write(cluster_map, 1)
print(f"Mapa de clusters: {cluster_tif_path}")

# Mapa de confianca
conf_tif_path = output_dir_maps / f'confidence_map_k{BEST_K}.tif'
with rasterio.open(
    conf_tif_path, 'w',
    driver='GTiff',
    height=height,
    width=width,
    count=1,
    dtype=np.float32,
    crs=crs,
    transform=transform,
    nodata=0,
    compress='lzw'
) as dst:
    dst.write(confidence_map, 1)
print(f"Mapa de confianca: {conf_tif_path}")

In [ ]:
# Visualizacao previa (recorte)
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Encontrar regiao com dados
valid_rows = rows[mask_conf]
valid_cols = cols[mask_conf]
r_min, r_max = valid_rows.min(), valid_rows.max()
c_min, c_max = valid_cols.min(), valid_cols.max()

# Margem
margin = 500
r_min = max(0, r_min - margin)
r_max = min(height, r_max + margin)
c_min = max(0, c_min - margin)
c_max = min(width, c_max + margin)

# Recorte
cluster_crop = cluster_map[r_min:r_max, c_min:c_max]
conf_crop = confidence_map[r_min:r_max, c_min:c_max]

# Plot clusters
ax1 = axes[0]
cmap_cluster = plt.cm.get_cmap('tab10', BEST_K + 2)
im1 = ax1.imshow(cluster_crop, cmap=cmap_cluster, vmin=-2, vmax=BEST_K-1)
ax1.set_title(f'Mapa de Clusters (K={BEST_K})')
cbar1 = plt.colorbar(im1, ax=ax1, shrink=0.7)
cbar1.set_ticks(range(-2, BEST_K))
cbar1.set_ticklabels(['Fora', 'Incerto'] + [f'C{i}' for i in range(BEST_K)])

# Plot confianca
ax2 = axes[1]
im2 = ax2.imshow(conf_crop, cmap='RdYlGn', vmin=0, vmax=1)
ax2.set_title('Mapa de Confianca')
plt.colorbar(im2, ax=ax2, shrink=0.7, label='Probabilidade')

plt.suptitle(f'{CLASSE_NOME} - Recorte', fontsize=14)
plt.tight_layout()
plt.savefig(output_dir_maps / 'preview.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Resumo

In [ ]:
import os

print("\n" + "="*60)
print("ARQUIVOS GERADOS")
print("="*60)

for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(str(OUTPUT_DIR), '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in sorted(files):
        filepath = os.path.join(root, file)
        size = os.path.getsize(filepath)
        if size > 1024*1024:
            size_str = f"{size/1024/1024:.1f} MB"
        elif size > 1024:
            size_str = f"{size/1024:.1f} KB"
        else:
            size_str = f"{size} B"
        print(f'{subindent}{file} ({size_str})')

In [ ]:
print("\n" + "="*60)
print("RESUMO FINAL")
print("="*60)
print(f"Classe: {CLASSE_NOME}")
print(f"Pixels processados: {len(ndvi):,}")
print(f"Melhor K: {BEST_K}")
print(f"Threshold: {BEST_THRESHOLD}")
print(f"Pixels confiaveis: {mask_conf.sum():,} ({mask_conf.mean()*100:.1f}%)")
print(f"\nPerfis dos clusters:")
for c in range(BEST_K):
    if c in profiles:
        p = profiles[c]
        print(f"  Cluster {c}: n={p['n_samples']:,}")
print(f"\nMapas salvos em: {output_dir_maps}")

---

## Conclusao

Clustering concluido!

**Arquivos gerados:**
- `samples.npz`: Dados NDVI extraidos
- `pretrained_model.pt`: Autoencoder pre-treinado
- `k{N}/`: Resultados para cada valor de K
- `output/`: Mapas GeoTIFF

**Proximos passos:**
- Repetir para outras classes
- Analisar padroes encontrados
- Usar clusters para refinamento de amostras